In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

In [3]:
os.environ["OPENAI_API_KEY"]= OPENAI_API_KEY

In [4]:
%pwd

'c:\\Users\\parvez\\Downloads\\Betopia pdf\\Betopia pdf'

In [5]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\parvez\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
file_path = "data/Document1.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()

In [7]:
data

[Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2026-02-11T00:42:24+06:00', 'author': 'parvez kabir', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-02-11T00:42:24+06:00', 'sourcemodified': "D:20260211004224+06'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/Document1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Betopia Group — Overview and Organizational Profile\nBetopia Group is a rapidly growing Bangladeshi multinational business and technology\nconglomerate focused on empowering people and organizations through innovation, digital\ntransformation, and enterprise solutions. The company operates with the belief that human\npotential is the strongest force for progress. By combining modern technology, strategic\nleadership, and collaborative culture, Betopia aims to build a future where businesses and\nindividuals can unlock limitless opportunities.\nAt its core, Betopia Group seeks to create

In [8]:
len(data)

4

In [9]:
question_gen = ""
for page in data:
    question_gen +=page.page_content

In [10]:
len(question_gen)

7039

In [11]:
!pip install -U langchain langchain-community langchain-openai

Defaulting to user installation because normal site-packages is not writeable


In [12]:
!pip install -U langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ekhon split koro
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_text(question_gen)

print(f"Success! Total chunks created: {len(chunks)}")

Success! Total chunks created: 9


In [14]:
!pip install -U langchain-core

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.11
    Uninstalling langchain-core-1.2.11:
      Successfully uninstalled langchain-core-1.2.11


In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document  # Ekhon eikhan theke import hobe

# 1. Chunks gulo-ke 'Document' object-e convert kora
docs = [Document(page_content=t) for t in chunks]

# 2. Embedding model initialize kora
embeddings = OpenAIEmbeddings()

# 3. Vector database toiri kora
vector_db = FAISS.from_documents(docs, embeddings)

print("Vector database ready without errors!")

Vector database ready without errors!


In [16]:
!pip install -U langchain

Defaulting to user installation because normal site-packages is not writeable


In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. LLM Setup
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# 2. Prompt Template (Jate AI sudhu boi-er tottho use kore)
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. RAG Chain Toiri
# 'vector_db' tumi agei toiri korecho, oitai use hobe
retriever = vector_db.as_retriever()

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Modern QA Chain Ready!")

Modern QA Chain Ready!


In [18]:
# Tomar proshno
query = "What is the mission of betopia ?"

# Result dekha
response = rag_chain.invoke(query)
print("Answer:", response)

Answer: The mission of Betopia Group is to enable businesses and communities to thrive in a digital-first world by delivering scalable, intelligent, and secure solutions.
